# Train football player/ball detector (Colab)

Parallel quality run to the local `scripts/train_player_detector.py` (yolov8m @1280). **Same data, same tight-ball GT, bigger model:** this notebook merges `football-players-detection-3zvbc` **v11** (298 train) + `minaehyeon/football-player-jrjtj` **v5** (465 train) exactly like `scripts/merge_player_datasets.py`, then fine-tunes **yolov8x @ imgsz=1280, batch=6, epochs=50**.

**Why no ball padding here:** the ball box is padded (+10px) only at inference (mirrors the Roboflow reference). Training uses tight GT so ball mAP stays standard and comparable to hosted v11 (92.9 mAP baseline). This makes the Colab run the third fair contender in the Stage-1 bake-off.

**Why 1280:** the ball is a small object (~8px at 1280 input from 1080p). At 640 detection accuracy collapses.

**Setup (once):**
1. Runtime -> Change runtime type -> GPU (T4 is fine).
2. Left **Secrets** panel -> add `ROBOFLOW_API_KEY` (app.roboflow.com -> Settings -> API Keys).
3. (Optional) Mount Drive so `best.pt` survives; otherwise download the weights before the session ends.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

v11 = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc').version(11)
minae = rf.workspace('minaehyeon').project('football-player-jrjtj').version(5)

d_v11 = v11.download('yolov8')
d_minae = minae.download('yolov8')
print('v11 at:', d_v11.location)
print('minae at:', d_minae.location)

In [ ]:
%%writefile merge_player_datasets.py
"""Copy of scripts/merge_player_datasets.py (tight GT, no ball padding)."""
import argparse
import shutil
from pathlib import Path

CANONICAL_CLASSES = ["ball", "goalkeeper", "player", "referee"]
SPLITS = ["train", "valid", "test"]


def read_class_names(data_yaml: Path) -> list[str]:
    """Parse names from Roboflow data.yaml (bullet or inline list format)."""
    names = []
    for line in data_yaml.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("names:"):
            inner = line[len("names:"):].strip()
            if inner.startswith("[") and inner.endswith("]"):
                inner = inner[1:-1]
            names.extend(t.strip().strip("'\"") for t in inner.split(",") if t.strip())
        elif line.startswith("-"):
            names.append(line[1:].strip().strip("'\""))
    return names


def merge(args) -> None:
    canonical = {n: i for i, n in enumerate(CANONICAL_CLASSES)}
    out = Path(args.out)
    for spec in args.source:
        tag, src = spec.split(":", 1)
        src = Path(src)
        names = read_class_names(src / "data.yaml")
        if not names:
            raise SystemExit(f"{tag}: no class names parsed from data.yaml")
        mapping = {n: canonical.get(n) for n in names}
        for split in SPLITS:
            img_dir = src / split / "images"
            if not img_dir.is_dir():
                continue
            for img in sorted(img_dir.glob("*")):
                if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                lab = img.with_suffix(".txt")
                if lab.parent.name != "labels":
                    lab = src / split / "labels" / (img.stem + ".txt")
                new_stem = f"{tag}__{img.stem}"
                dst_img = out / split / "images" / (new_stem + img.suffix)
                dst_lab = out / split / "labels" / (new_stem + ".txt")
                dst_img.parent.mkdir(parents=True, exist_ok=True)
                dst_lab.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(img, dst_img)
                lines = []
                if lab.is_file():
                    for raw in lab.read_text().splitlines():
                        parts = raw.split()
                        if not parts:
                            continue
                        try:
                            cls_name = names[int(parts[0])]
                        except (ValueError, IndexError):
                            continue
                        cid = mapping.get(cls_name)
                        if cid is not None:
                            lines.append(f"{cid} " + " ".join(parts[1:]))
                dst_lab.write_text("\n".join(lines))

    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\n"
        f"train: train/images\n"
        f"val: valid/images\n"
        f"test: test/images\n\n"
        f"names: {CANONICAL_CLASSES}\n"
        f"nc: {len(CANONICAL_CLASSES)}\n"
    )
    print(f"merged data.yaml written to {out / 'data.yaml'}")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--out", required=True)
    p.add_argument("--source", action="append", required=True)
    merge(p.parse_args())

In [ ]:
import subprocess
subprocess.run(["python", "merge_player_datasets.py",
                "--out", "/content/player_detection_merged",
                "--source", f"v11:{d_v11.location}",
                "--source", f"minae:{d_minae.location}"], check=True)
print(open('/content/player_detection_merged/data.yaml').read())

In [ ]:
# PHASE A - Fine-tuning (quality recipe: yolov8x @1280, batch 6).
# Same tight-ball GT as the local run; expect ~1h on a T4.
!yolo task=detect mode=train model=yolov8x.pt \
    data=/content/player_detection_merged/data.yaml \
    batch=6 epochs=50 imgsz=1280 plots=True project=/content/runs name=merged

In [ ]:
from IPython.display import Image
Image(filename='/content/runs/merged/results.png', width=600)
Image(filename='/content/runs/merged/confusion_matrix.png', width=600)

In [ ]:
# Validate on the merged test split at the training imgsz (bake-off metric).
!yolo task=detect mode=val model=/content/runs/merged/weights/best.pt \
    data=/content/player_detection_merged/data.yaml imgsz=1280 split=test

### Save + deploy

Run the next two cells **before the Colab session dies** (weights live in the VM only).

In [ ]:
# PHASE B (optional) - Download weights to Drive for local inference.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/PerformanceAnalyzer/models
!cp /content/runs/merged/weights/best.pt /content/drive/MyDrive/PerformanceAnalyzer/models/player_colab_x.pt
print('copied to Drive: MyDrive/PerformanceAnalyzer/models/player_colab_x.pt')

In [ ]:
# PHASE B (optional) - Deploy on Roboflow for the hosted endpoint.
# Free tier allows one deployed model at a time; this is what created the
# model we currently use via ROBOFLOW_PLAYER_MODEL_ID.
v11.deploy(model_type='yolov8', model_path='/content/runs/merged/')
print('deployed. Point ROBOFLOW_PLAYER_MODEL_ID at the new version.')